# 04. Model Analysis & Improvement Planning

## 목적
- 클래스 분포 정확히 파악
- SHAP 기반 피처 기여도 분석
- Threshold별 성능 변화 확인
- 개선 방향 도출

## 입력
- `train.csv`, `test.csv`
- 학습된 XGBoost 모델 (`.pkl`)

## 출력
- 클래스 분포 통계
- SHAP summary plot
- Threshold-Performance 곡선

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

print("=== 04. Model Analysis 시작 ===")

## 설정

In [ ]:
# 경로 설정
DATA_DIR = '../../data-pipeline/data/processed/all_features'
MODEL_DIR = '../models/lightgbm/v1'
OUTPUT_DIR = '../outputs/lightgbm/v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 타겟 정의
TARGETS = {
    'death': 'death_next_24h',
    'vent': 'vent_next_12h',
    'pressor': 'pressor_next_12h',
    'composite': 'composite_next_24h'
}

TIME_WINDOWS = {
    'death': '24h',
    'vent': '12h',
    'pressor': '12h',
    'composite': '24h'
}

## Step 1: 데이터 및 모델 로드

In [ ]:
# 데이터 로드
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

with open(os.path.join(DATA_DIR, 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)

X_train = train[feature_cols]
X_test = test[feature_cols]

print(f"Train: {len(train):,} rows")
print(f"Test: {len(test):,} rows")
print(f"Features: {len(feature_cols)}개")

In [ ]:
# 모델 로드
models = {}
for name in TARGETS.keys():
    time_window = TIME_WINDOWS[name]
    model_path = os.path.join(MODEL_DIR, time_window, f'{name}.pkl')
    with open(model_path, 'rb') as f:
        models[name] = pickle.load(f)
    print(f"✓ {name} 모델 로드 완료")

## Step 2: 클래스 분포 분석

In [ ]:
print("=" * 60)
print(" 클래스 분포 (Train Set)")
print("=" * 60)

class_dist = []
for name, col in TARGETS.items():
    pos_count = train[col].sum()
    neg_count = len(train) - pos_count
    pos_rate = train[col].mean() * 100
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else np.inf
    
    class_dist.append({
        'Target': name,
        'Positive': f"{pos_count:,}",
        'Negative': f"{neg_count:,}",
        'Pos %': f"{pos_rate:.2f}%",
        'Imbalance (Neg:Pos)': f"{imbalance_ratio:.1f}:1"
    })
    
    print(f"\n[{name.upper()}] {col}")
    print(f"  Positive: {pos_count:,} ({pos_rate:.2f}%)")
    print(f"  Negative: {neg_count:,} ({100-pos_rate:.2f}%)")
    print(f"  Imbalance Ratio: {imbalance_ratio:.1f}:1")

# 테이블로 정리
class_dist_df = pd.DataFrame(class_dist)
print("\n")
print(class_dist_df.to_string(index=False))

In [ ]:
# 시각화
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for idx, (name, col) in enumerate(TARGETS.items()):
    ax = axes[idx]
    counts = train[col].value_counts().sort_index()
    colors = ['#3498db', '#e74c3c']
    bars = ax.bar(['Negative', 'Positive'], counts.values, color=colors)
    ax.set_title(f'{name}\n({train[col].mean()*100:.2f}% positive)', fontsize=11)
    ax.set_ylabel('Count')
    
    # 값 표시
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000, 
                f'{count:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 저장: ../../data-pipeline/reports/class_distribution.png")

## Step 3: Threshold별 성능 분석

In [ ]:
def analyze_thresholds(y_true, y_prob, thresholds=[0.1, 0.2, 0.3, 0.4, 0.5]):
    """다양한 threshold에서 성능 계산"""
    results = []
    for thresh in thresholds:
        y_pred = (y_prob >= thresh).astype(int)
        results.append({
            'Threshold': thresh,
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0),
            'Predicted Pos': y_pred.sum(),
            'Actual Pos': y_true.sum()
        })
    return pd.DataFrame(results)

# 각 모델별 threshold 분석
print("=" * 70)
print(" Threshold별 성능 분석")
print("=" * 70)

threshold_results = {}
for name, col in TARGETS.items():
    y_true = test[col]
    y_prob = models[name].predict_proba(X_test)[:, 1]
    
    result_df = analyze_thresholds(y_true, y_prob, [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5])
    threshold_results[name] = result_df
    
    print(f"\n[{name.upper()}]")
    print(result_df.to_string(index=False))

In [ ]:
# Precision-Recall 곡선 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

colors = {'death': '#e74c3c', 'vent': '#3498db', 'pressor': '#2ecc71', 'composite': '#9b59b6'}

for idx, (name, col) in enumerate(TARGETS.items()):
    ax = axes[idx]
    y_true = test[col]
    y_prob = models[name].predict_proba(X_test)[:, 1]
    
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    
    ax.plot(recall, precision, color=colors[name], lw=2)
    ax.set_xlabel('Recall', fontsize=11)
    ax.set_ylabel('Precision', fontsize=11)
    ax.set_title(f'{name} - PR Curve', fontsize=12, fontweight='bold')
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3)
    
    # Baseline (random)
    baseline = y_true.mean()
    ax.axhline(y=baseline, color='gray', linestyle='--', label=f'Baseline: {baseline:.3f}')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pr_curves_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 저장: ../outputs/xgboost/v1/pr_curves_analysis.png")

## Step 4: SHAP 분석

In [ ]:
# SHAP 설치 확인
try:
    import shap
    print(f"SHAP version: {shap.__version__}")
except ImportError:
    print("SHAP 설치 필요: pip install shap")

In [ ]:
import shap

# 샘플링 (메모리 절약)
SAMPLE_SIZE = 2000
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=min(SAMPLE_SIZE, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]

print(f"SHAP 분석 샘플: {len(X_sample):,}개")

In [ ]:
# Composite 모델 SHAP (대표)
print("\n[COMPOSITE] SHAP 분석 중...")
explainer = shap.TreeExplainer(models['composite'])
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
plt.title('Feature Importance (SHAP) - Composite Model', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_composite_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 저장: ../outputs/xgboost/v1/shap_composite_bar.png")

In [ ]:
# SHAP Summary Plot (방향성 포함)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title('SHAP Summary - Composite Model', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_composite_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 저장: ../outputs/xgboost/v1/shap_composite_summary.png")

In [ ]:
# 4개 모델 비교 (Top 10 피처)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, model) in enumerate(models.items()):
    ax = axes[idx]
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # 피처 중요도 계산
    shap_importance = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': shap_importance
    }).sort_values('importance', ascending=True).tail(15)
    
    ax.barh(importance_df['feature'], importance_df['importance'], color=colors[name])
    ax.set_title(f'{name.upper()} - Top 15 Features (SHAP)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_all_models.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ 저장: ../outputs/xgboost/v1/shap_all_models.png")

## Step 5: 오분류 분석 (False Negatives)

In [ ]:
print("=" * 60)
print(" False Negative 분석 (놓친 Positive 케이스)")
print("=" * 60)

for name, col in TARGETS.items():
    y_true = test[col].values
    y_prob = models[name].predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    
    # False Negatives
    fn_mask = (y_true == 1) & (y_pred == 0)
    fn_count = fn_mask.sum()
    total_pos = y_true.sum()
    
    # True Positives
    tp_mask = (y_true == 1) & (y_pred == 1)
    
    print(f"\n[{name.upper()}]")
    print(f"  Total Positive: {total_pos:,}")
    print(f"  Caught (TP): {tp_mask.sum():,} ({tp_mask.sum()/total_pos*100:.1f}%)")
    print(f"  Missed (FN): {fn_count:,} ({fn_count/total_pos*100:.1f}%)")
    
    # FN 케이스의 예측 확률 분포
    fn_probs = y_prob[fn_mask]
    if len(fn_probs) > 0:
        print(f"  FN 예측확률: mean={fn_probs.mean():.3f}, max={fn_probs.max():.3f}")

In [ ]:
# FN vs TP 피처 비교 (Composite)
y_true = test['composite_next_24h'].values
y_prob = models['composite'].predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

fn_mask = (y_true == 1) & (y_pred == 0)
tp_mask = (y_true == 1) & (y_pred == 1)

fn_features = X_test[fn_mask].mean()
tp_features = X_test[tp_mask].mean()

comparison = pd.DataFrame({
    'FN (Missed)': fn_features,
    'TP (Caught)': tp_features,
    'Diff (TP-FN)': tp_features - fn_features
}).round(3)

print("\n[COMPOSITE] FN vs TP 피처 평균 비교 (차이 큰 순)")
print(comparison.sort_values('Diff (TP-FN)', key=abs, ascending=False).head(15))

## Step 6: 요약 및 개선 방향

In [ ]:
print("\n" + "=" * 70)
print(" 04. Model Analysis 완료")
print("=" * 70)

print("""
📊 분석 결과 요약:

1. 클래스 분포
   - 결과 확인 후 작성

2. Threshold 분석
   - 결과 확인 후 작성
   
3. SHAP 분석
   - Top 피처 확인 후 작성

4. 오분류 분석
   - FN 패턴 확인 후 작성

📁 저장된 파일:
   - ../outputs/xgboost/v1/class_distribution.png
   - ../outputs/xgboost/v1/pr_curves_analysis.png
   - ../outputs/xgboost/v1/shap_composite_bar.png
   - ../outputs/xgboost/v1/shap_composite_summary.png
   - ../outputs/xgboost/v1/shap_all_models.png
""")